In [20]:
# Install AI Platform SDK
!pip install --upgrade google-cloud-aiplatform

# Imports
from google.cloud import aiplatform
import pandas as pd

# Initialize AI Platform (your API key is already configured in AI Studio)
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "us-central1"
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Example papers DataFrame
papers_df = pd.DataFrame({
    "title": [
        "Learning From Failure: Integrating Negative Experiences",
        "Cognitive Architectures for Language Agents",
        "Demystifying Instruction Mixing for Fine-tuning",
        "WizardLM: Empowering large pre-trained language models",
        "Mobile-Agent: Autonomous Multi-Modal Mobile Devices"
    ],
    "summary": [
        "Large language models (LLMs) have achieved success...",
        "Recent efforts have augmented large language models...",
        "Instruction tuning significantly enhances the...",
        "Training large language models (LLMs) with open-source data...",
        "Mobile device agent based on Multimodal Large Language Models..."
    ]
})

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 53.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.1
    Uninstalling cachetools-6.2.1:
      Successfully uninstalled cachetools-6.2.1
  Attempting uninstall: google-cloud-aiplatform
    Found existing installation: google-cloud-aiplatform 1.125.0
    Uninstalling google-cloud-aiplatform-1.125.0:
      Successfully uninstalled google-cloud-aiplatform-1.125.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 req

In [21]:
# Step 1: Search Agent - Fetch papers from ArXiv
import requests
import pandas as pd
from xml.etree import ElementTree

def search_arxiv(query: str, max_results: int = 5):
    """
    Search ArXiv papers using a keyword query and return a DataFrame.
    
    Args:
        query (str): The research topic to search for.
        max_results (int): Number of papers to fetch.
        
    Returns:
        pd.DataFrame: DataFrame with columns: title, summary, published, link
    """
    url = f'http://export.arxiv.org/api/query?search_query=all:{query}&start=0&max_results={max_results}'
    response = requests.get(url)
    
    if response.status_code != 200:
        raise Exception(f"ArXiv API request failed with status {response.status_code}")
    
    # Parse XML
    root = ElementTree.fromstring(response.content)
    ns = {'arxiv': 'http://www.w3.org/2005/Atom'}
    
    papers = []
    for entry in root.findall('arxiv:entry', ns):
        title = entry.find('arxiv:title', ns).text.strip()
        summary = entry.find('arxiv:summary', ns).text.strip()
        published = entry.find('arxiv:published', ns).text.strip()
        link = entry.find('arxiv:id', ns)
        if link is None:
            link = entry.find('arxiv:link', ns).attrib.get('href', '')
        else:
            link = link.text.strip()
        
        papers.append({
            'title': title,
            'summary': summary,
            'published': published,
            'link': link
        })
    
    df = pd.DataFrame(papers)
    return df

# Example usage
topic = "large language models"
papers_df = search_arxiv(topic, max_results=5)

# Check results
papers_df

,title,summary,published,link
0,Large Language Models Lack Understanding of Ch...,Large language models (LLMs) have demonstrated...,2024-05-18T18:08:58Z,http://arxiv.org/abs/2405.11357v3
1,Is Self-knowledge and Action Consistent or Not...,"In this study, we delve into the validity of c...",2024-02-22T16:32:08Z,http://arxiv.org/abs/2402.14679v2
2,Unmasking the Shadows of AI: Investigating Dec...,This research critically navigates the intrica...,2024-02-07T00:21:46Z,http://arxiv.org/abs/2403.09676v1
3,Self-Cognition in Large Language Models: An Ex...,While Large Language Models (LLMs) have achiev...,2024-07-01T17:52:05Z,http://arxiv.org/abs/2407.01505v1
4,Making Large Language Models Better Reasoners ...,Reasoning is a cognitive process of using evid...,2023-09-05T11:32:48Z,http://arxiv.org/abs/2309.02144v1


In [26]:
# Install the official Google Generative AI Python client
!pip install --quiet google-generativeai

In [32]:
import google.generativeai as genai

# Set the API key (optional in AI Studio)
genai.configure(api_key="GOOGLE_API_KEY")

# Summarize function
def summarize_paper(text: str) -> str:
    response = genai.generate(
        model="text-bison-001",
        prompt=f"Summarize the following research abstract into 3-5 key points:\n\n{text}",
        temperature=0.2,
        max_output_tokens=300
    )
    # Get the generated text
    return response.text